# Pertemuan 05 — Estimasi Densitas dan Gaussian Mixture Model (GMM)

Pada modul ini kita akan mempelajari **Gaussian Mixture Model (GMM)** beserta cara melatihnya menggunakan **algoritma Expectation-Maximization (EM)**.

> **Kaitan dengan pertemuan sebelumnya.** Pada Pertemuan 04 kita memakai k-means, yang menetapkan setiap titik ke **tepat satu** klaster. GMM adalah pengembangannya: setiap titik diberi **probabilitas** untuk tiap klaster. Sebuah titik yang berada di perbatasan bisa 60% milik klaster A dan 40% milik klaster B, bukan dipaksa memilih salah satu. Pengelompokan seperti ini disebut **soft clustering**.

### Apa yang akan kita kerjakan

| Bagian | Isi |
|---|---|
| 1 | Distribusi Gaussian satu dimensi dan multivariat |
| 2 | GMM dengan scikit-learn pada data bersepeda |
| 3 | Ancestor sampling: membangkitkan data dari sebuah GMM |
| 4 | Membangun sendiri algoritma EM (inisialisasi, E-step, M-step) |
| 5 | Masalah pada estimasi maksimum likelihood GMM |

> **Cara memakai modul ini.** Jalankan sel kode berurutan dari atas ke bawah. Beberapa sel memakai `ipywidgets` sehingga hanya berfungsi di lingkungan Jupyter, bukan pada tampilan statis.

**Prasyarat:** memahami k-means (Pertemuan 04) dan konsep probabilitas bersyarat serta Teorema Bayes.


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly import figure_factory as ff
from plotly.subplots import make_subplots
colors = px.colors.qualitative.Plotly
px.defaults.width = 800
from ipywidgets import HBox
import numpy as np
pd.set_option('plotting.backend', 'plotly')

In [ ]:
# Membuat folder 'images' bila belum ada
import os
if not os.path.exists("images"):
    os.makedirs("images")

In [ ]:
### Hapus tanda pagar bila ingin mengekspor ke HTML
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

---
## 1. Distribusi Gaussian

Sebelum membahas campuran (*mixture*), kita perlu mengenal komponen penyusunnya lebih dahulu.

Fungsi kepadatan peluang (*probability density function*, PDF) dari distribusi Gaussian **satu peubah** dengan rata-rata $\mu$ dan ragam $\sigma^2$ adalah:

$$
f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \, e^{-\frac{(x-\mu)^2}{2\sigma^2}}
$$

**Cara membacanya:**

* Faktor $\frac{1}{\sqrt{2\pi\sigma^2}}$ adalah **konstanta penormal** yang memastikan luas seluruh kurva sama dengan 1.
* Bagian $e^{-(x-\mu)^2 / 2\sigma^2}$ menyatakan bahwa kepadatan **menurun secara eksponensial** seiring menjauhnya $x$ dari rata-rata $\mu$.
* Ragam $\sigma^2$ mengatur lebar kurva: semakin besar ragamnya, semakin melebar dan mendatar kurvanya.

> **Catatan istilah.** Pada `scipy.stats.norm`, argumen `loc` berarti rata-rata dan `scale` berarti **simpangan baku** ($\sigma$), bukan ragam. Karena itu pada kode di bawah kita menuliskan `np.sqrt(variance)`.


In [ ]:
from scipy.stats import norm
mean = 0
variance = .7
x = np.linspace(-4, 4, 100)
p = norm.pdf(x, loc=mean, scale=np.sqrt(variance))  # loc = rata-rata, scale = simpangan baku
fig = px.line(x=x, y=p, title=f"Distribusi Normal (rata-rata={mean}, ragam={variance})",
        labels={"x": "x", "y": "p(x)"}, width = 700, height = 400)
# fig.write_image("images/standard_normal.pdf", scale=2, height=400, width=700)
fig


Kurva di atas adalah bentuk lonceng yang sudah dikenal. Cobalah ubah nilai `variance` menjadi 0,1 lalu 3, dan amati bagaimana kurvanya menyempit atau melebar sementara luas di bawahnya tetap sama dengan 1.

### 1.1 Distribusi Normal Multivariat

Data kita biasanya lebih dari satu dimensi. Untuk itu kita perlu bentuk Gaussian berdimensi banyak:

$$
f(\mathbf{x}) = \frac{1}{\sqrt{(2\pi)^D|\Sigma|}} \exp\left(-\frac{1}{2}(\mathbf{x}-\mu)^T\Sigma^{-1}(\mathbf{x}-\mu)\right)
$$

dengan $\mathbf{x}$ adalah vektor acak berdimensi $D$, $\mu$ adalah **vektor rata-rata**, dan $\Sigma$ adalah **matriks kovariansi**.

**Peran matriks kovariansi $\Sigma$:**

| Isi $\Sigma$ | Bentuk sebaran |
|---|---|
| Diagonal sama besar | Lingkaran |
| Diagonal berbeda, luar diagonal nol | Elips sejajar sumbu |
| Ada nilai di luar diagonal | Elips miring (kedua peubah berkorelasi) |

Bagian $(\mathbf{x}-\mu)^T\Sigma^{-1}(\mathbf{x}-\mu)$ disebut **jarak Mahalanobis kuadrat**. Inilah bedanya dengan k-means yang memakai jarak Euclidean biasa: jarak Mahalanobis memperhitungkan sebaran dan korelasi data, sehingga klasternya boleh berbentuk elips miring, bukan hanya bulat.

Sel berikut menghitung sendiri rumus di atas, lalu kita bandingkan hasilnya dengan `scipy` untuk memastikan perhitungannya benar.


In [ ]:
def mv_normal_pdf(X, mu, Sigma):
    """Menghitung kepadatan normal multivariat pada titik-titik X."""
    d = X.shape[1]
    X_centered = X - mu
    Sigma_inv = np.linalg.inv(Sigma)
    norm_const = 1 / np.sqrt((2 * np.pi) ** d * np.linalg.det(Sigma))
    exp_term = np.exp(-0.5 * np.sum(X_centered @ Sigma_inv * X_centered, axis=1))
    return norm_const * exp_term

In [ ]:
mu = np.array([1, 0])
Sigma = np.array([[3, 0.4], [0.4, 2]])

from scipy.stats import multivariate_normal
normal = multivariate_normal(mean=mu, cov=Sigma)
normal.pdf(np.array([[1, 0.5]]))

In [ ]:
# Bandingkan dengan hasil scipy di atas — keduanya harus sama
mv_normal_pdf(np.array([[1, 0.5]]), mu, Sigma)

Kedua hasil sama, artinya rumus yang kita tulis sendiri sudah benar. Menyandingkan implementasi sendiri dengan pustaka baku seperti ini adalah kebiasaan yang baik saat mempelajari algoritma.

In [ ]:
def plot_bivariate_normal(mu, Sigma, fig=None):
    from scipy.stats import multivariate_normal
    normal = multivariate_normal(mean=mu, cov=Sigma)
    u = np.linspace(-9, 9, 100)
    X = np.array(np.meshgrid(u,u)).reshape(2,-1).T
    Z = normal.pdf(X)
    if fig is None:
        fig = make_subplots(rows=1, cols=2,
                            specs=[[{'type': 'surface'}, {'type': 'contour'}]],)
    fig.add_surface(x=X[:,0].reshape(100,100), y=X[:,1].reshape(100,100),
                    z=Z.reshape(100,100), colorscale='Viridis',
                    contours=dict(z=dict(show=True, size=.01, start=0, end=0.3)), row=1, col=1)
    fig.add_contour(x=u, y=u, z=Z.reshape(100,100), colorscale='Viridis',
                    line_smoothing=1.3,
                    #contours_coloring='lines',
                    showscale=False,
                    row=1, col=2
                    )
    fig.update_layout(width=900, height=500)
    return fig

In [ ]:
mu = np.array([1, 0])
Sigma = np.array([[3, 0.4], [0.4, 2]])
plot_bivariate_normal(mu, Sigma)

Panel kiri menampilkan kepadatannya sebagai permukaan tiga dimensi, panel kanan menampilkannya sebagai peta kontur — dua cara melihat objek yang sama.

Perhatikan bahwa kontur berbentuk **elips miring**. Kemiringan itu berasal dari nilai 0,4 pada posisi luar diagonal matriks $\Sigma$, yang menyatakan adanya korelasi positif antara kedua peubah.

### 1.2 Plot Interaktif

Plot interaktif berikut **hanya berfungsi di lingkungan Jupyter Notebook**, tidak pada tampilan statis atau hasil ekspor PDF.

Geser kelima penggeser (*slider*) dan amati perubahannya:

* `mu1` dan `mu2` — menggeser **posisi** pusat sebarannya.
* `sigma11` dan `sigma22` — mengubah **lebar** sebaran pada masing-masing sumbu.
* `sigma12` — mengubah **kemiringan** elips. Cobalah beri nilai positif besar, lalu negatif besar, dan amati arah kemiringannya berbalik.


In [ ]:
from ipywidgets import interactive_output, FloatSlider, HBox, VBox, widgets

u = np.linspace(-9, 9, 100)
X = np.array(np.meshgrid(u,u)).reshape(2,-1).T
normal = multivariate_normal(mean=mu, cov=Sigma)
Z = normal.pdf(X)
fig1 = go.FigureWidget()
fig1.add_surface(x=X[:,0].reshape(100,100), y=X[:,1].reshape(100,100),
                z=Z.reshape(100,100), colorscale='Viridis',
                contours=dict(z=dict(show=True, size=.01, start=0, end=0.3)))
fig1.update_layout(width=600, height=500)
fig2 = go.FigureWidget()
fig2.add_contour(x=u, y=u, z=Z.reshape(100,100), colorscale='Viridis',
                line_smoothing=1.3)
fig2.update_layout(width=400, height=500)

mu1 = FloatSlider(min=-5, max=5, step=0.1, value=1, description='mu1')
mu2 = FloatSlider(min=-5, max=5, step=0.1, value=0, description='mu2')
sigma11 = FloatSlider(min=0.1, max=5, step=0.1, value=3, description='sigma11')
sigma22 = FloatSlider(min=0.1, max=5, step=0.1, value=2, description='sigma22')
sigma12 = FloatSlider(min=-3, max=3, step=0.1, value=0.4, description='sigma12')


def update(mu1, mu2, sigma11, sigma22, sigma12):
    mu = np.array([mu1, mu2])
    Sigma = np.array([[sigma11, sigma12], [sigma12, sigma22]])
    normal = multivariate_normal(mean=mu, cov=Sigma)
    Z = normal.pdf(X).reshape(100,100)
    with fig1.batch_update():
        fig1.data[0].z = Z
    with fig2.batch_update():
        fig2.data[0].z = Z

interactive_output(update, {
    'mu1': mu1, 'mu2': mu2,
    'sigma11': sigma11, 'sigma22': sigma22, 'sigma12': sigma12
})

HBox([VBox([mu1, mu2, sigma11, sigma22, sigma12]), fig1, fig2],
     layout=widgets.Layout(align_items='center'))

> **Sebelum lanjut.** Pastikan Anda paham bahwa satu Gaussian hanya mampu menggambarkan **satu** gerombolan berbentuk elips. Untuk data yang memiliki beberapa gerombolan, kita membutuhkan **campuran** beberapa Gaussian — itulah GMM.

---
*Kembali ke slide.*

---


---
## 2. Dataset Bersepeda

Seperti pada pertemuan sebelumnya, kita memakai data perjalanan bersepeda untuk mengilustrasikan konsepnya. Data ini berisi **kecepatan** (*Speed*) dan **panjang lintasan** (*Length*) dari perjalanan yang dilakukan dengan beberapa sepeda berbeda.

Memakai data yang sama seperti Pertemuan 04 memudahkan kita membandingkan langsung hasil k-means dengan hasil GMM.


In [ ]:
# bikes = pd.read_csv("speed_length_data.csv")
bikes = pd.read_csv("https://eecs189.org/fa25/resources/assets/lectures/lec04/speed_length_data.csv")
bikes.head()

In [ ]:
bikes.plot.scatter(x='Speed', y='Length', title='Kecepatan vs Panjang Segmen Perjalanan',
                   height=800)

Sekilas terlihat ada beberapa gerombolan dengan bentuk yang **memanjang dan tidak seragam**. Inilah alasan GMM sering lebih cocok daripada k-means untuk data semacam ini: k-means cenderung menghasilkan klaster berbentuk bulat, sedangkan GMM dapat menyesuaikan bentuk elipsnya.

---
## 3. Gaussian Mixture Model

**Gaussian Mixture Model (GMM)** adalah model probabilistik yang mengasumsikan bahwa seluruh titik data dibangkitkan dari **campuran beberapa distribusi Gaussian**:

$$
p(x \, \vert \, \pi, \mu, \Sigma) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k)
$$

**Arti setiap parameter:**

| Simbol | Arti | Syarat |
|---|---|---|
| $K$ | Banyaknya komponen Gaussian | ditentukan sendiri, seperti `k` pada k-means |
| $\pi_k$ | Bobot campuran komponen ke-$k$ | semuanya positif dan berjumlah 1 |
| $\mu_k$ | Vektor rata-rata komponen ke-$k$ | — |
| $\Sigma_k$ | Matriks kovariansi komponen ke-$k$ | — |

**Cara membacanya secara intuitif.** Bayangkan proses dua langkah: pertama kita undi sebuah komponen berdasarkan bobot $\pi$, lalu kita bangkitkan satu titik dari Gaussian komponen terpilih itu. Data yang kita amati adalah hasil pengulangan proses ini berkali-kali.

Sama seperti k-means, kita dapat memakai kelas `GaussianMixture` dari `sklearn.mixture` untuk melatih GMM pada data kita.

> **Perhatikan `predict_proba`.** Inilah keunggulan utama GMM dibanding k-means. Selain memberikan nomor klaster lewat `predict`, GMM juga memberikan **derajat keyakinan** untuk setiap klaster. Kolom `prob` pada tabel berikut berisi probabilitas tertinggi bagi setiap titik — semakin kecil nilainya, semakin ragu model terhadap penempatan titik itu.


In [ ]:
from sklearn.mixture import GaussianMixture
# Membuat Gaussian Mixture Model dengan 4 komponen
gmm = GaussianMixture(n_components=4, random_state=42, )
# Melatih model pada data (tanpa label y, sama seperti k-means)
gmm.fit(bikes[['Speed', 'Length']])
# Mengambil nomor klaster dan probabilitas tertingginya
bikes['scikit gmm'] = gmm.predict(bikes[['Speed', 'Length']]).astype(str)
bikes['prob'] = gmm.predict_proba(bikes[['Speed', 'Length']]).max(axis=1)
bikes

In [ ]:
mu = gmm.means_
Sigma = [np.linalg.inv(p) for p in gmm.precisions_]
p = gmm.weights_

In [ ]:
def gmm_surface(mu, Sigma, p, u_pts, v_pts):
    from scipy.stats import multivariate_normal
    u, v = np.meshgrid(u_pts, v_pts)
    X_pts = np.array([u.flatten(), v.flatten()]).T
    Z = np.zeros(X_pts.shape[0])
    for k in range(len(p)):
        Z += p[k] * multivariate_normal(mu[k], Sigma[k]).pdf(X_pts)
    return go.Contour(x=u_pts, y=v_pts, z=Z.reshape(u.shape),
                      colorscale='Viridis',
                      colorbar=dict(x=1.05, y=0.35, len=0.75)
                    )

In [ ]:
num_points = 100
speed_pts = np.linspace(bikes['Speed'].min()-3, bikes['Speed'].max()+3, num_points)
length_pts = np.linspace(bikes['Length'].min()-3, bikes['Length'].max()+3, num_points)

fig = go.Figure()
fig.add_trace(gmm_surface(mu, Sigma, p, speed_pts, length_pts))
fig.update_layout(width=800, height=800)
fig.add_traces(px.scatter(bikes, x='Speed', y='Length', color='scikit gmm').data)

Grafik di atas menampilkan kepadatan campuran hasil pelatihan sebagai peta kontur, dengan titik data digambar di atasnya.

Perhatikan bahwa daerah kontur yang rapat menandakan kepadatan tinggi, dan bentuk setiap gundukan mengikuti sebaran titik di sekitarnya — inilah yang tidak dapat dilakukan k-means.

In [ ]:
fig = px.scatter(bikes, x='Speed', y='Length', symbol='scikit gmm',
           size='prob', color="scikit gmm", title='Hasil Pengelompokan GMM (ukuran penanda = tingkat keyakinan)',
           color_continuous_scale="Viridis_r", size_max=15)
fig.update_layout(width=800, height=800)

> **Amati grafik terakhir.** Ukuran penanda mencerminkan tingkat keyakinan model. Penanda yang kecil menunjukkan titik yang berada di daerah perbatasan antar-komponen — di situlah GMM memberikan informasi yang tidak bisa diberikan k-means.

---
*Kembali ke slide.*

---


---
## 4. Ancestor Sampling pada GMM

**Ancestor sampling** adalah cara membangkitkan data buatan dari sebuah model probabilistik dengan mengikuti urutan sebab-akibatnya. Untuk GMM, prosesnya dua langkah:

1. **Undi komponennya.** Pilih $z_n \sim \text{Categorical}(\pi)$, yaitu tentukan titik ini berasal dari komponen mana.
2. **Bangkitkan titiknya.** Ambil $x_n \sim \mathcal{N}(\mu_{z_n}, \Sigma_{z_n})$ dari Gaussian komponen terpilih.

Variabel $z$ disebut **variabel laten** (*latent variable*): ia menentukan data yang kita amati, tetapi pada data nyata nilainya tidak pernah kita ketahui. Justru karena $z$ tersembunyi inilah kita membutuhkan algoritma EM nanti.

Di sini kita membangkitkan 100 titik satu dimensi dari campuran tiga Gaussian. Karena kita sendiri yang membuat datanya, kita **tahu** nilai $z$ yang sebenarnya — dan itu berguna untuk memeriksa apakah algoritma kita bekerja benar.


In [ ]:
# Ancestor sampling untuk membangkitkan data buatan
np.random.seed(42)
N = 100

mu = np.array([-1, 2, 5])
pi = np.array([0.2, 0.5, 0.3])
Sigma = np.array([0.2, 0.5, .1])

In [ ]:
# Langkah 1: undi komponennya (variabel laten z)
# Langkah 2: bangkitkan titik dari Gaussian komponen terpilih
z = np.random.choice(len(mu), size=N, p=pi)
x = np.random.normal(mu[z], np.sqrt(Sigma[z]))

In [ ]:
# Log-likelihood: ukuran kecocokan parameter dengan data
log_likelihood = np.sum(np.log(np.sum(
    pi[z] * norm.pdf(x[:, None], loc=mu[z], scale=np.sqrt(Sigma[z])),
    axis=1 )))

In [ ]:
# Diurutkan agar tampilannya lebih mudah dibaca
ind = z.argsort()
z = z[ind]
x = x[ind]

fig = px.scatter(x=x, y=np.random.rand(N)/20,
                 title=f'Data Buatan dari GMM (Log-Likelihood: {log_likelihood:.2f})',
                 opacity = 0.7,
                 color=z.astype(str), labels={'color': 'Komponen asli'}, height=400)
u = np.linspace(-4, 9, 1000)
df = pd.DataFrame({'x': u})
for k in range(len(mu)):
    df[f'p{k}'] = pi[k] * norm.pdf(u, loc=mu[k], scale=np.sqrt(Sigma[k]))
df['p'] = df[[f'p{k}' for k in range(len(mu))]].sum(axis=1)
fig.add_traces(px.line(df, x='x', y=df.columns[1:], labels={'y': 'Kepadatan'}).data)
fig.update_layout(width=800, height=400)

fig

> **Amati grafik di atas.** Kurva berwarna adalah ketiga komponen Gaussian, masing-masing sudah dikalikan bobot $\pi_k$-nya. Kurva hitam adalah jumlah ketiganya, yaitu kepadatan campuran secara keseluruhan. Perhatikan bahwa komponen dengan bobot lebih besar menghasilkan kurva yang lebih tinggi dan titik data yang lebih rapat.

Nilai **log-likelihood** yang tertera pada judul mengukur seberapa cocok parameter model dengan data. Semakin besar nilainya (semakin mendekati nol dari arah negatif), semakin baik kecocokannya. Angka inilah yang akan berusaha dimaksimalkan oleh algoritma EM.

---
*Kembali ke slide.*

---


---
## 5. Membangun GMM dengan Algoritma EM

Pada data nyata kita hanya melihat $x$, sedangkan komponen asalnya ($z$) tersembunyi. Kita ingin mencari parameter yang memaksimalkan log-likelihood:

\begin{align*}
\log p\left(\mathcal{D} \,\vert\, \mu, \Sigma \right)
& = \log \left( \prod_{n=1}^{N} \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x_n \mid \mu_k, \Sigma_k) \right) \\
& = \sum_{n=1}^{N} \log \left(\sum_{k=1}^{K} \pi_k \, \mathcal{N}(x_n \mid \mu_k, \Sigma_k) \right)
\end{align*}

**Mengapa ini sulit dimaksimalkan secara langsung?** Karena ada **penjumlahan di dalam logaritma**. Logaritma dari sebuah hasil kali dapat dipecah menjadi jumlah logaritma sehingga mudah diturunkan, tetapi logaritma dari sebuah penjumlahan tidak bisa dipecah. Akibatnya tidak ada rumus tertutup untuk mencari parameter optimalnya.

**Gagasan EM.** Kalau saja kita tahu nilai $z$ setiap titik, persoalannya menjadi mudah: cukup hitung rata-rata dan kovariansi tiap kelompok. Karena $z$ tidak diketahui, EM menempuh jalan berputar dan bergantian:

| Langkah | Yang dikerjakan | Analoginya |
|---|---|---|
| **E-step** | Dengan parameter saat ini, hitung peluang setiap titik berasal dari tiap komponen | "Seandainya parameternya benar, siapa milik siapa?" |
| **M-step** | Dengan peluang tadi, hitung ulang parameter yang paling cocok | "Seandainya kepemilikannya benar, berapa parameter terbaiknya?" |

Kedua langkah diulang bergantian. Setiap putaran dijamin **tidak menurunkan** nilai log-likelihood, sehingga algoritma pasti konvergen — meskipun belum tentu ke solusi terbaik secara global.

> **Hubungan dengan k-means.** K-means sebenarnya adalah bentuk khusus EM yang disederhanakan: penugasannya bersifat tegas (0 atau 1, bukan probabilitas), dan kovariansinya dianggap bulat serta sama untuk semua klaster.


### 5.1 Langkah Inisialisasi

Cara yang lazim untuk memulai GMM adalah menjalankan **k-means** terlebih dahulu, lalu memakai pusat klasternya sebagai rata-rata awal setiap komponen Gaussian.

Parameter awal lainnya disetel sederhana:

* $\Sigma_k$ diawali dengan **matriks identitas** — anggapan awal bahwa setiap komponen berbentuk lingkaran satuan.
* $\pi_k$ diawali sama rata, yaitu $1/K$ untuk setiap komponen.

> **Mengapa tidak diawali acak sepenuhnya?** EM peka terhadap titik awal. Memulai dari hasil k-means membuat algoritma bermula dari posisi yang sudah masuk akal, sehingga lebih cepat konvergen dan lebih kecil kemungkinannya terjebak pada solusi yang buruk.


In [ ]:
from sklearn.cluster import KMeans
def initialize_gmm(x, K):
    N, D = x.shape
    kmeans = KMeans(n_clusters=K)
    kmeans.fit(x)
    mu = kmeans.cluster_centers_
    Sigma = np.array([np.eye(D) for _ in range(K)])
    p = np.ones(K) / K
    return mu, Sigma, p

In [ ]:
# mu berasal dari pusat k-means, Sigma diawali matriks identitas, p diawali sama rata
mu, Sigma, p = initialize_gmm(bikes[['Speed', 'Length']], 4)
display(mu, Sigma, p)

### 5.2 Langkah Ekspektasi (E-step)

Pada E-step kita menghitung **responsibility**, yaitu peluang setiap titik data berasal dari setiap komponen Gaussian, berdasarkan parameter yang berlaku saat ini.

Secara formal, kita menghitung probabilitas posterior dengan Teorema Bayes:

$$
\gamma_{nk} = p(z_n = k \mid x_n) = \frac{\pi_k \, \mathcal{N}(x_n \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \, \mathcal{N}(x_n \mid \mu_j, \Sigma_j)}
$$

**Membaca kodenya:**

1. Untuk setiap komponen $k$, hitung $\pi_k \cdot \mathcal{N}(x \mid \mu_k, \Sigma_k)$ — inilah pembilangnya.
2. Bagi setiap baris dengan jumlah seluruh barisnya agar total tiap baris menjadi 1 — inilah penormalannya.

Hasilnya berupa matriks berukuran $N \times K$: setiap baris mewakili satu titik data, setiap kolom mewakili satu komponen, dan isinya adalah peluang.

> **Di sinilah letak "soft" pada soft clustering.** Bandingkan dengan k-means yang hanya menghasilkan satu nomor klaster per titik. Di sini setiap titik memperoleh sebaran peluang atas seluruh komponen.


In [ ]:
def E_step(x, mu, Sigma, p):
    """Langkah E pada algoritma EM.

    Menghitung probabilitas posterior variabel laten (z) bila data diketahui.
    Hasilnya matriks N x K: baris = titik data, kolom = komponen.
    """
    N, D = x.shape
    K = len(p)
    assert(Sigma.shape == (K, D, D))
    assert(mu.shape == (K,D))
    p_z_given_x = np.zeros((N, K))
    for k in range(K):
        p_z_given_x[:, k] = p[k] * multivariate_normal(mu[k], Sigma[k]).pdf(x)
    p_z_given_x /= p_z_given_x.sum(axis=1, keepdims=True)  # dinormalkan agar tiap baris berjumlah 1
    return p_z_given_x


In [ ]:
# Hasilnya matriks N x K: satu baris per titik data, satu kolom per komponen
p_z_given_x = E_step(bikes[['Speed', 'Length']], mu, Sigma, p)
p_z_given_x.shape

In [ ]:
p_z_given_x.sum(axis=1)  # Setiap baris seharusnya berjumlah 1

Setiap baris berjumlah 1, seperti seharusnya. Ini bukti bahwa penormalan pada E-step berjalan benar: setiap titik data membagi keyakinannya ke seluruh komponen, dan totalnya selalu 100%.

### 5.3 Langkah Maksimisasi (M-step)

Pada langkah ini kita memperbarui parameter setiap komponen Gaussian berdasarkan responsibility yang baru saja dihitung.

Rumusnya menyerupai perhitungan rata-rata dan kovariansi biasa, hanya saja **berbobot** menurut responsibility:

$$
N_k = \sum_{n=1}^{N} \gamma_{nk}, \qquad
\mu_k = \frac{1}{N_k}\sum_{n=1}^{N} \gamma_{nk}\, x_n, \qquad
\pi_k = \frac{N_k}{N}
$$

$$
\Sigma_k = \frac{1}{N_k}\sum_{n=1}^{N} \gamma_{nk}\,(x_n - \mu_k)(x_n - \mu_k)^T
$$

**Cara memahaminya.** $N_k$ dapat dibaca sebagai "jumlah titik efektif yang dimiliki komponen $k$". Sebuah titik yang 70% milik komponen 1 menyumbang 0,7 titik kepada komponen itu dan 0,3 kepada komponen lain. Setelah itu, $\mu_k$ tak lain adalah rata-rata berbobot, dan $\pi_k$ adalah proporsi kepemilikan komponen tersebut.

> **Perhatikan baris regularisasi.** Baris `Sigma_new += 1e-3 * np.eye(D)` menambahkan nilai kecil pada diagonal matriks kovariansi. Tanpa itu, sebuah komponen bisa menyusut sampai kovariansinya nyaris nol dan matriksnya tidak dapat dibalik — persoalan yang akan kita bahas pada Bagian 6.


In [ ]:
def M_step(x, p_z_given_x):
    """Langkah M pada algoritma EM.

    Memperbarui parameter (mu, Sigma, p) berdasarkan probabilitas posterior.
    Semua rumusnya adalah rata-rata berbobot dengan bobot berupa responsibility.
    """
    N, D = x.shape
    N, K = p_z_given_x.shape
    mu_new = np.zeros((K, D))
    Sigma_new = np.zeros((K, D, D))
    p_new = np.zeros(K)

    for k in range(K):
        N_k = p_z_given_x[:, k].sum()
        mu_new[k, :] = p_z_given_x[:, k] @ x / N_k
        Sigma_new[k, :, :] = (p_z_given_x[:, k] * (x - mu_new[k, :]).T @ (x - mu_new[k, :])) / N_k
        Sigma_new[k, :, :] += 1e-3 * np.eye(D)  # regularisasi: mencegah kovariansi menyusut ke nol
        p_new[k] = N_k / N

    return mu_new, Sigma_new, p_new


In [ ]:
M_step(bikes[['Speed', 'Length']], p_z_given_x)

Bandingkan `mu_new` dengan `mu` awal sebelumnya — posisinya sudah bergeser mengikuti responsibility.

Satu putaran E-step diikuti M-step disebut **satu iterasi EM**. Berikutnya kita ulang keduanya sampai parameternya stabil.

In [ ]:
def em_algorithm(x, K, max_iters=100, initial_variance=100):
    D = 2
    p = np.ones(K) / K
    # Ambil rata-rata awal dari hasil k-means, lalu diberi guncangan acak
    mu, Sigma, p = initialize_gmm(x, K)
    mu = mu + np.random.randn(*mu.shape) * 3
    soln_path = [(mu, Sigma, p)]
    for i in range(max_iters):
        p_z_given_x = E_step(x, mu, Sigma, p)
        mu, Sigma, p = M_step(x, p_z_given_x)
        soln_path.append((mu, Sigma, p))
    return mu, Sigma, p, soln_path

In [ ]:
# Menjalankan EM sebanyak 50 iterasi pada data sepeda
mu, Sigma, p, soln_path = em_algorithm(bikes[['Speed', 'Length']].values,
                                       K=4,
                                       max_iters=50)
print("mu", mu)
print("Sigma", Sigma)

In [ ]:
num_points = 100
speed_pts = np.linspace(bikes['Speed'].min()-3, bikes['Speed'].max()+3, num_points)
length_pts = np.linspace(bikes['Length'].min()-3, bikes['Length'].max()+3, num_points)

mu, Sigma, p = soln_path[-1]
fig = go.Figure()
fig.add_trace(gmm_surface(mu, Sigma, p, speed_pts, length_pts))
fig.update_layout(width=800, height=800)
fig.add_traces(px.scatter(bikes, x='Speed', y='Length', color='scikit gmm').data)
fig.add_scatter(x=mu[:,0], y=mu[:,1], mode='markers', marker=dict(color='black', size=10), name='Pusat komponen')

Bandingkan hasil implementasi kita ini dengan hasil `GaussianMixture` dari scikit-learn pada bagian 3. Bentuk konturnya seharusnya sangat mirip, meskipun urutan penomoran komponennya bisa berbeda.

> **Catatan.** Penomoran klaster pada pengelompokan tidak punya makna intrinsik. Klaster "0" pada satu kali proses bisa menjadi klaster "2" pada proses berikutnya, tanpa mengubah kualitas hasilnya.

In [ ]:
from ipywidgets import  IntSlider
np.random.seed(42)
mu, Sigma, p, soln_path = em_algorithm(bikes[['Speed', 'Length']].values,
                                       K=4,
                                       max_iters=100)
num_points = 100
speed_pts = np.linspace(bikes['Speed'].min()-3, bikes['Speed'].max()+3, num_points)
length_pts = np.linspace(bikes['Length'].min()-3, bikes['Length'].max()+3, num_points)

mu, Sigma, p = soln_path[0]
fig = go.FigureWidget()
fig.add_trace(gmm_surface(mu, Sigma, p, speed_pts, length_pts))
fig.update_layout(width=800, height=800)
fig.add_traces(px.scatter(bikes, x='Speed', y='Length', color='scikit gmm').data)
fig.add_scatter(x=mu[:,0], y=mu[:,1], mode='markers', marker=dict(color='black', size=10), name='Pusat komponen')

def update(step):
    mu, Sigma, p = soln_path[step]
    with fig.batch_update():
        fig.data[0].z = gmm_surface(mu, Sigma, p, speed_pts, length_pts).z
    with fig.batch_update():
        fig.data[-1].x = mu[:, 0]
        fig.data[-1].y = mu[:, 1]
step_slider = IntSlider(min=0, max=len(soln_path)-1, step=1, value=0, description='Iterasi')
interactive_output(update, {'step': step_slider})
VBox([fig, step_slider])



Geser penggeser **Iterasi** dari 0 sampai akhir, dan amati bagaimana kontur kepadatan berubah bentuk dari lingkaran-lingkaran awal menjadi elips yang mengikuti sebaran data.

> **Perhatikan.** Perubahan terbesar terjadi pada beberapa iterasi pertama, lalu melambat — pola yang sama seperti pada k-means di Pertemuan 04.

---
## 6. Masalah pada Estimasi Maksimum Likelihood GMM

Ada satu kelemahan mendasar pada GMM yang perlu Anda ketahui: **fungsi likelihood-nya tidak terbatas** (*unbounded*).

**Apa maksudnya?** Bila sebuah komponen Gaussian dipusatkan tepat pada satu titik data, lalu ragamnya dikecilkan menuju nol, kepadatan pada titik itu membesar menuju tak hingga. Akibatnya nilai log-likelihood juga menuju tak hingga.

Dengan kata lain, ada "solusi" yang secara matematis memberikan likelihood tak hingga, tetapi sama sekali tidak berguna: komponen itu hanya menghafal satu titik data dan tidak menjelaskan apa pun. Ini adalah bentuk ekstrem dari **overfitting**.

Sel-sel berikut menunjukkan gejalanya: kita bandingkan parameter yang benar dengan parameter ekstrem yang ragamnya dibuat sangat kecil ($10^{-100}$), lalu bandingkan nilai log-likelihood keduanya.

**Cara mengatasinya dalam praktik:**

| Cara | Penjelasan |
|---|---|
| Regularisasi kovariansi | Menambahkan nilai kecil pada diagonal $\Sigma$, seperti pada M-step kita |
| Batas bawah ragam | Scikit-learn menyediakan argumen `reg_covar` untuk ini |
| Pendekatan Bayesian | Memberi *prior* pada parameter, misalnya lewat `BayesianGaussianMixture` |
| Menjalankan berulang | Memakai beberapa inisialisasi lalu memilih hasil yang wajar |

> **Pelajaran yang lebih luas.** Nilai likelihood tertinggi tidak selalu berarti model terbaik. Prinsip yang sama muncul kembali pada regularisasi model linear dan pada *early stopping* di deep learning.


In [ ]:
# Ancestor sampling untuk membangkitkan data buatan
np.random.seed(42)
N = 100

mu = np.array([-1, 2, 5])
pi = np.array([0.2, 0.5, 0.3])
Sigma = np.array([0.2, 0.5, .1])

z = np.random.choice(len(mu), size=N, p=pi)
x = np.random.normal(mu[z], np.sqrt(Sigma[z]))

log_likelihood = np.sum(np.log(np.sum(
    pi[z] * norm.pdf(x[:, None], loc=mu[z], scale=np.sqrt(Sigma[z])),
    axis=1 )))

In [ ]:
# Diurutkan agar tampilannya lebih mudah dibaca
ind = z.argsort()
z = z[ind]
x = x[ind]

fig = px.scatter(x=x, y=np.random.rand(N)/20,
                 title=f'Data Buatan dari GMM (Log-Likelihood: {log_likelihood:.2f})',
                 opacity = 0.7,
                 color=z.astype(str), labels={'color': 'Komponen asli'}, height=400)
u = np.linspace(-4, 9, 1000)
df = pd.DataFrame({'x': u})
for k in range(len(mu)):
    df[f'p{k}'] = pi[k] * norm.pdf(u, loc=mu[k], scale=np.sqrt(Sigma[k]))
fig.add_traces(px.line(df, x='x', y=df.columns[1:], labels={'y': 'Kepadatan'}).data)
fig.update_layout(width=800, height=400)

fig

Grafik di atas menampilkan parameter yang **benar**, yaitu parameter yang memang dipakai membangkitkan datanya. Catat nilai log-likelihood-nya pada judul, lalu bandingkan dengan sel berikutnya.

In [ ]:
# Parameter ekstrem: dua komponen dipusatkan pada titik terluar dengan ragam nyaris nol
mu = np.array([x.min(), x.mean(), x.max()])
Sigma = np.array([1e-100, 10, 1e-100])
pi = np.array([0.3, 0.4, 0.3])

log_likelihood = np.sum(np.log(np.sum(
    pi[z] * norm.pdf(x[:, None], loc=mu[z], scale=np.sqrt(Sigma[z])),
    axis=1 )))

fig = px.scatter(x=x, y=np.random.rand(N)/20,
                 title=f'Parameter Ekstrem pada GMM (Log-Likelihood: {log_likelihood:.2f})',
                 opacity = 0.7,
                 color=z.astype(str), labels={'color': 'Komponen asli'}, height=400)
u = np.linspace(-4, 9, 100)
u = np.append(u, mu)
u.sort()
df = pd.DataFrame({'x': u})
for k in range(len(mu)):
    df[f'p{k}'] = pi[k] * norm.pdf(u, loc=mu[k], scale=np.sqrt(Sigma[k]))
fig.add_traces(px.line(df, x='x', y=df.columns[1:], labels={'y': 'Kepadatan'}).data)
fig.update_layout(width=800, height=400)
fig.update_layout(yaxis_range=[0, 1])

fig

**Bandingkan nilai log-likelihood kedua grafik.** Parameter ekstrem ini menghasilkan log-likelihood yang jauh lebih tinggi daripada parameter yang sebenarnya benar — padahal modelnya jelas tidak masuk akal: dua komponennya hanya "menghafal" satu titik data masing-masing.

Inilah bukti nyata bahwa memaksimalkan likelihood begitu saja dapat menyesatkan. Kalau ragam boleh mengecil tanpa batas, likelihood dapat dibuat setinggi apa pun tanpa model menjadi lebih baik.

---
## 7. Penutup

### Ringkasan

**Distribusi Gaussian**

* Gaussian satu peubah ditentukan oleh rata-rata $\mu$ dan ragam $\sigma^2$.
* Gaussian multivariat ditentukan oleh vektor rata-rata $\mu$ dan matriks kovariansi $\Sigma$; unsur di luar diagonal $\Sigma$ membuat elipsnya miring.

**Gaussian Mixture Model**

* GMM memodelkan data sebagai campuran $K$ Gaussian dengan bobot $\pi_k$ yang berjumlah 1.
* Berbeda dengan k-means yang menetapkan satu klaster secara tegas, GMM memberi **probabilitas** keanggotaan (*soft clustering*) melalui `predict_proba`.
* Karena memakai matriks kovariansi, klaster GMM boleh berbentuk **elips miring**, bukan hanya bulat seperti pada k-means.

**Algoritma EM**

* Log-likelihood GMM sulit dimaksimalkan langsung karena ada penjumlahan di dalam logaritma.
* EM menyiasatinya secara bergantian: **E-step** menghitung responsibility $\gamma_{nk}$, **M-step** memperbarui $\mu_k$, $\Sigma_k$, dan $\pi_k$ dengan rumus rata-rata berbobot.
* Setiap iterasi tidak pernah menurunkan log-likelihood, tetapi hasil akhirnya bergantung pada inisialisasi.
* K-means dapat dipandang sebagai bentuk sederhana dari EM dengan penugasan tegas dan kovariansi bulat.

**Keterbatasan**

* Likelihood GMM tidak terbatas: sebuah komponen dapat menyusut ke satu titik dan menghasilkan likelihood tak hingga.
* Karena itu regularisasi kovariansi bukan sekadar penambah kestabilan numerik, melainkan keharusan.

### Latihan Mandiri

1. Pada plot interaktif bagian 1.2, setel `sigma12` menjadi 0. Bentuk apa yang dihasilkan? Lalu naikkan perlahan menuju 3. Apa arti nilai `sigma12` bagi hubungan kedua peubah?
2. Bandingkan hasil `GaussianMixture` dengan `KMeans` pada data sepeda yang sama. Adakah titik yang dikelompokkan berbeda oleh keduanya? Menurut Anda mengapa?
3. Urutkan data sepeda berdasarkan kolom `prob` dari yang terkecil, lalu tampilkan sepuluh baris teratas. Di posisi seperti apa titik-titik paling tidak meyakinkan itu berada pada grafik pencar?
4. Jalankan `em_algorithm` beberapa kali **tanpa** menyetel `np.random.seed`. Apakah hasil akhirnya selalu sama? Kaitkan jawaban Anda dengan sifat EM yang hanya menjamin optimum lokal.
5. Ubah `max_iters` pada `em_algorithm` menjadi 1, 3, 10, dan 50. Pada iterasi keberapa hasilnya mulai terlihat stabil?
6. Hapus baris regularisasi `Sigma_new += 1e-3 * np.eye(D)` pada `M_step`, lalu jalankan ulang algoritmanya beberapa kali. Galat apa yang muncul, dan mengapa?
7. Coba nilai `K` = 2, 3, 5, dan 6 pada `GaussianMixture`, lalu bandingkan nilai `gmm.bic(X)` untuk masing-masing. Nilai `K` mana yang memberikan BIC terkecil? Bacalah dokumentasinya untuk memahami apa yang diukur oleh BIC.

### Bacaan Lanjutan

* Dokumentasi Gaussian Mixture scikit-learn: https://scikit-learn.org/stable/modules/mixture.html
* Perbandingan GMM dan k-means: https://scikit-learn.org/stable/modules/clustering.html
